# BI-Prefix Degradation Analysis

Primary question: given one fixed simple replacement operator, how do layer selection and the number of simultaneous replacements affect model degradation?

This notebook is a new maintained experiment. The historical MVP provides motivation only; none of its archived measurements are merged into the results below. BI is treated as a selection baseline, while the replacement operator remains frozen.

## Fixed experimental choices

- Dense bias-free hidden-size linear replacement, matching the B4 baseline.
- Every eligible layer is fitted once from dense-model activations.
- All subset evaluations reuse those frozen replacements.
- No recovery.
- Layers 0 and 23 are screened but protected from replacement.
- The k experiments are prefix searches, not exhaustive subset searches.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

from mlp_replacement.capture import collect_modules_io
from mlp_replacement.config import DataConfig, ModelConfig
from mlp_replacement.data import build_data_loaders
from mlp_replacement.degradation import (
    build_degradation_selections, degradation_interaction,
)
from mlp_replacement.evaluation.language_model import evaluate_language_model
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import discover_mlp_blocks, load_model_and_tokenizer
from mlp_replacement.operators import fit_ridge_linear
from mlp_replacement.runlog import environment_record, json_value
from mlp_replacement.screening import compute_bi_scores
from mlp_replacement.selection import eligible_layer_indices
from mlp_replacement.surgery import temporary_replacements

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
SEED = 21
RIDGE = 1e-4
K_VALUES = (1, 2, 3, 4, 5, 6)
RANDOM_SEEDS = (11, 21, 31, 41, 51, 61, 71, 81, 91, 101)
PROTECTED_PREFIX = 1
PROTECTED_SUFFIX = 1
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'results' / 'notebook-block-study' / 'degradation-bi-prefix.json'

model_config = ModelConfig(model_id='HuggingFaceTB/SmolLM2-1.7B', device='auto', dtype='auto')
data_config = DataConfig(
    sequence_length=128, batch_size=2,
    num_calibration_batches=48, num_operator_validation_batches=24,
    num_recovery_batches=0, num_recovery_validation_batches=0,
    num_model_validation_batches=24, num_test_batches=0, seed=SEED,
)
torch.manual_seed(SEED)

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_config)
device = next(model.parameters()).device
loaders = build_data_loaders(tokenizer, data_config, include_recovery=False)
refs = discover_mlp_blocks(model)
refs_by_index = {ref.index: ref for ref in refs}
available_indices = tuple(ref.index for ref in refs)
eligible_indices = eligible_layer_indices(
    available_indices, PROTECTED_PREFIX, PROTECTED_SUFFIX
)
print({'device': str(device), 'layers': available_indices, 'eligible': eligible_indices})

## Two explicitly named BI scopes

transformer_layer_bi is the canonical residual-stream metric. mlp_sublayer_bi is an adapted MLP input/output metric retained for continuity with the MVP. The adapted score is never presented as canonical BI.

In [ ]:
canonical_bi = compute_bi_scores(
    model, loaders.calibration, data_config.num_calibration_batches, device,
    scope='transformer_layer',
)
adapted_bi = compute_bi_scores(
    model, loaders.calibration, data_config.num_calibration_batches, device,
    scope='mlp_sublayer',
)
bi_df = pd.DataFrame({
    'layer': available_indices,
    'transformer_layer_bi': [canonical_bi.scores[index] for index in available_indices],
    'mlp_sublayer_bi': [adapted_bi.scores[index] for index in available_indices],
    'eligible': [index in eligible_indices for index in available_indices],
})
canonical_ranks = bi_df['transformer_layer_bi'].rank()
adapted_ranks = bi_df['mlp_sublayer_bi'].rank()
bi_spearman = float(canonical_ranks.corr(adapted_ranks))
print({'spearman': bi_spearman})
bi_df

In [ ]:
bi_long = bi_df.melt(id_vars=['layer', 'eligible'], value_vars=['transformer_layer_bi', 'mlp_sublayer_bi'], var_name='metric', value_name='score')
plt.figure(figsize=(14, 5))
sns.lineplot(data=bi_long, x='layer', y='score', hue='metric', marker='o')
plt.axvspan(-0.5, 0.5, color='grey', alpha=0.15, label='protected')
plt.axvspan(22.5, 23.5, color='grey', alpha=0.15)
plt.title(f'BI profiles across all layers; rank correlation = {bi_spearman:.3f}')
plt.tight_layout()

## Fit one frozen dense-linear replacement per eligible layer

All activation pairs are captured from the untouched dense model in one pass per data split. Each layer is solved once with the same relative ridge convention used in baseline-testing.ipynb.

In [ ]:
eligible_paths = [refs_by_index[index].path for index in eligible_indices]
capture_kwargs = dict(device=device, storage_device='cpu', storage_dtype=torch.float32)
training_by_path = collect_modules_io(
    model, eligible_paths, loaders.calibration, data_config.num_calibration_batches, **capture_kwargs
)
validation_by_path = collect_modules_io(
    model, eligible_paths, loaders.operator_validation, data_config.num_operator_validation_batches, **capture_kwargs
)


In [ ]:
replacements = {}
local_rows = []
for position, index in enumerate(eligible_indices, start=1):
    path = refs_by_index[index].path
    replacement = fit_ridge_linear(
        training_by_path[path], ridge=RIDGE, bias=False, device=device
    ).to(device)
    local = evaluate_operator(replacement, validation_by_path[path], device, batch_size=2048)
    replacements[index] = replacement
    local_rows.append({
        'layer': index, 'depth_fraction': index / (len(available_indices) - 1),
        'local_mse': local.mse,
        'local_relative_mse': local.relative_mse, 'local_r2': local.r2,
        'local_cosine': local.cosine_similarity,
    })
    print(f'Fitted layer {index} ({position}/{len(eligible_indices)})')
local_df = pd.DataFrame(local_rows)
del training_by_path, validation_by_path
if torch.cuda.is_available():
    torch.cuda.empty_cache()
local_df

In [ ]:
baseline_lm = evaluate_language_model(
    model, loaders.model_validation, device, data_config.num_model_validation_batches
)

def evaluate_subset(indices):
    indices = tuple(sorted(indices))
    with temporary_replacements(model, {index: replacements[index] for index in indices}) as manifest:
        metrics = evaluate_language_model(
            model, loaders.model_validation, device, data_config.num_model_validation_batches
        )
    return {
        'indices': list(indices), 'k': len(indices),
        'loss': metrics.loss, 'perplexity': metrics.perplexity,
        'delta_loss': metrics.loss - baseline_lm.loss,
        'delta_perplexity': metrics.perplexity - baseline_lm.perplexity,
        'replacement_parameters': sum(record.replacement_parameters for record in manifest.records),
        'removed_parameters': manifest.removed_parameters,
    }

baseline_lm

## Single-layer degradation scan

This separates local approximation error, BI, and depth from the integrated effect of replacing one layer.

In [ ]:
singleton_rows = []
for index in eligible_indices:
    row = evaluate_subset((index,))
    row.update({'strategy': 'single_layer', 'seed': None, 'layer': index})
    singleton_rows.append(row)
singleton_df = (
    pd.DataFrame(singleton_rows)
    .merge(local_df, on='layer')
    .merge(bi_df, on='layer')
)
singleton_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(21, 11))
sns.lineplot(data=singleton_df, x='layer', y='delta_loss', marker='o', ax=axes[0, 0])
sns.scatterplot(data=singleton_df, x='transformer_layer_bi', y='local_relative_mse', hue='layer', palette='viridis', ax=axes[0, 1])
sns.scatterplot(data=singleton_df, x='mlp_sublayer_bi', y='local_relative_mse', hue='layer', palette='viridis', ax=axes[0, 2], legend=False)
sns.scatterplot(data=singleton_df, x='transformer_layer_bi', y='delta_loss', hue='layer', palette='viridis', ax=axes[1, 0], legend=False)
sns.scatterplot(data=singleton_df, x='mlp_sublayer_bi', y='delta_loss', hue='layer', palette='viridis', ax=axes[1, 1], legend=False)
sns.scatterplot(data=singleton_df, x='local_relative_mse', y='delta_loss', hue='layer', palette='viridis', ax=axes[1, 2], legend=False)
axes[0, 0].set_title('Single-layer model degradation across depth')
axes[0, 1].set_title('Canonical BI versus local error')
axes[0, 2].set_title('MLP-local BI versus local error')
axes[1, 0].set_title('Canonical BI versus degradation')
axes[1, 1].set_title('MLP-local BI versus degradation')
axes[1, 2].set_title('Local error versus model degradation')
plt.tight_layout()

## Multi-block prefix experiment

Evaluate k from 1 through 6 for low/high canonical BI, low/high MLP-local BI, and ten deterministic random-order prefixes. Reusing a random seed gives nested subsets as k grows. Observed degradation is compared with the additive prediction from singleton runs.

In [ ]:
selections = build_degradation_selections(
    canonical_bi.scores, adapted_bi.scores, eligible_indices, K_VALUES, RANDOM_SEEDS
)
selection_df = pd.DataFrame([asdict(item) for item in selections])
selection_df['indices'] = selection_df['indices'].apply(list)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(selection_df[['strategy', 'k', 'seed', 'indices']])

In [ ]:
singleton_loss = dict(zip(singleton_df['layer'], singleton_df['delta_loss']))
subset_rows = []
for selection in selections:
    row = evaluate_subset(selection.indices)
    row.update({'strategy': selection.strategy, 'seed': selection.seed})
    row['additive_delta_loss'] = sum(singleton_loss[index] for index in selection.indices)
    row['interaction_delta_loss'] = degradation_interaction(
        row['delta_loss'], selection.indices, singleton_loss
    )
    subset_rows.append(row)
subset_df = pd.DataFrame(subset_rows)
subset_df

In [ ]:
membership_rows = []
for row_index, row in subset_df.iterrows():
    label = f"{row['strategy']}|k={row['k']}|seed={row['seed']}"
    membership_rows.append({
        'label': label,
        **{f'L{index}': int(index in row['indices']) for index in eligible_indices},
    })
membership_df = pd.DataFrame(membership_rows).set_index('label')
plt.figure(figsize=(16, max(8, len(membership_df) * 0.18)))
sns.heatmap(membership_df, cmap='Blues', cbar=False, linewidths=0.1)
plt.title('Exact layer membership for every evaluated subset')
plt.tight_layout()

In [ ]:
bi_subset_df = subset_df.query("strategy != 'random'")
random_summary = subset_df.query("strategy == 'random'").groupby('k')['delta_loss'].agg(['mean', 'min', 'max']).reset_index()
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
sns.lineplot(data=bi_subset_df, x='k', y='delta_loss', hue='strategy', marker='o', ax=axes[0])
axes[0].plot(random_summary['k'], random_summary['mean'], color='black', marker='o', label='random mean')
axes[0].fill_between(random_summary['k'], random_summary['min'], random_summary['max'], color='grey', alpha=0.25, label='random range')
axes[0].set_title('Degradation versus number of replacements')
axes[0].legend()
sns.scatterplot(data=subset_df, x='removed_parameters', y='delta_loss', hue='strategy', alpha=0.75, ax=axes[1])
axes[1].set_title('Degradation versus removed parameters')
sns.scatterplot(data=subset_df, x='additive_delta_loss', y='delta_loss', hue='strategy', alpha=0.75, ax=axes[2], legend=False)
limits = [min(subset_df['additive_delta_loss'].min(), subset_df['delta_loss'].min()), max(subset_df['additive_delta_loss'].max(), subset_df['delta_loss'].max())]
axes[2].plot(limits, limits, color='black', linestyle='--')
axes[2].set_title('Observed versus additive prediction')
plt.tight_layout()

## Reserved advanced-operator comparison

After baseline-testing identifies a stronger equal-cost operator, fit it independently for the required layers and evaluate only k in {1, 3, 6} for canonical-low, canonical-high, and random seed 21. Do not populate this section until the operator family and exact parameter budget are frozen.

In [ ]:
print('Advanced comparison is deferred until baseline-testing identifies a stronger equal-cost operator.')

In [ ]:
def json_records(frame):
    clean = frame.astype(object).where(pd.notna(frame), None)
    return clean.to_dict(orient='records')

artifact = {
    'schema_version': 1,
    'status': 'exploratory',
    'experiment': 'dense-linear-bi-prefix-degradation',
    'model': {
        'id': model_config.model_id,
        'resolved_revision': getattr(model.config, '_commit_hash', None),
        'requested_config': asdict(model_config),
    },
    'data': {
        'sequence_length': data_config.sequence_length,
        'calibration_batches': data_config.num_calibration_batches,
        'operator_validation_batches': data_config.num_operator_validation_batches,
        'model_validation_batches': data_config.num_model_validation_batches,
        'seed': SEED,
        'requested_config': asdict(data_config),
    },
    'environment': environment_record(),
    'operator': {'kind': 'dense_linear', 'bias': False, 'ridge': RIDGE, 'recovery': False},
    'protected_prefix': PROTECTED_PREFIX,
    'protected_suffix': PROTECTED_SUFFIX,
    'k_values': list(K_VALUES), 'random_seeds': list(RANDOM_SEEDS),
    'bi_spearman': bi_spearman,
    'bi': json_records(bi_df),
    'baseline_language_model': asdict(baseline_lm),
    'singletons': json_records(singleton_df),
    'subsets': json_records(subset_df),
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(json_value(artifact), indent=2, allow_nan=False), encoding='utf-8')
print(f'Saved exact selections and degradation results to {OUTPUT_PATH}')

## Interpretation boundary

These curves describe one model, one calibration distribution, one fixed dense-linear operator, and prefix-based subsets. They can reveal failures of simple BI selection and non-additive error propagation, but they cannot establish that low-BI layers are universally replaceable.